# MediFlow 2차 확장 적응

**GPU 선택 → 모두 실행 → Drive 연결**을 합니다. 기존 `mediflow_Project/MediFlow_Adaptation_Common_v3.zip`와
1차 선택 기록을 읽습니다. GPU 런타임 하나에서 실행하세요.

피부 기본·얼굴 선정·두피 선정 모델의 **1차 학습 전 저장본**에서 각각 5회 학습합니다.
1차와 같은 학습 사진과 조명 범위에 측정한 색·미세 대비·약한 흐림을 추가합니다.
검증은 원본·조명 변환·확장 변환으로 나뉩니다. 완료하면 학습 전·1차·2차 모델의
점수를 세 검증 조건에서 비교합니다. 결과는 탐색 비교로 기록합니다.

결과는 `내 드라이브/MediFlow_adaptation_runs/4ae6e07eb4af1225/stage2`에 저장합니다.
화면에 현재 분야, `Epoch 1/5`, 학습 배치, 검증과 저장 단계가 표시됩니다.
끊기면 같은 노트북을 다시 실행해 마지막 완료 epoch부터 이어갑니다.

마지막 셀은 저장한 기록으로 학습곡선을 만들고 Drive에 PNG로 저장합니다. 2차를 이미 완료했다면 마지막 셀만 실행하세요.


In [ ]:
import hashlib
import json
import os
import shutil
import signal
import stat
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path, PurePosixPath

from google.colab import drive


def digest(path):
    value = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            value.update(chunk)
    return value.hexdigest()


def unpack_bundle(bundle, destination, expected_hash):
    """Verify the previously supplied bundle before using any of its Python files."""
    if digest(bundle) != expected_hash:
        raise ValueError(
            "선택한 ZIP이 준비된 실행 ZIP과 다릅니다. MediFlow_Adaptation_Common_v3.zip을 선택하세요."
        )
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(bundle) as archive:
        manifest = json.loads(archive.read("bundle_manifest.json"))
        members = {}
        for info in archive.infolist():
            name = info.orig_filename
            parts = PurePosixPath(name).parts
            if (
                not parts
                or PurePosixPath(name).is_absolute()
                or ".." in parts
                or "\\" in name
                or ":" in name
                or stat.S_ISLNK(info.external_attr >> 16)
                or name.casefold() in members
            ):
                raise ValueError("안전하지 않거나 중복된 ZIP 경로: " + name)
            members[name.casefold()] = info
        files = {info.filename for info in members.values() if not info.is_dir()}
        if files != set(manifest) | {"bundle_manifest.json"}:
            raise ValueError("실행 ZIP의 파일 목록이 검증 목록과 다릅니다.")
        for name, expected in manifest.items():
            body = archive.read(name)
            if hashlib.sha256(body).hexdigest() != expected:
                raise ValueError("실행 파일 검사 실패: " + name)
            path = destination / name
            if not path.resolve().is_relative_to(destination):
                raise ValueError("ZIP 경로가 실행 폴더 밖을 가리킵니다.")
            path.parent.mkdir(parents=True, exist_ok=True)
            with path.open("xb") as stream:
                stream.write(body)
    return destination


drive.mount("/content/drive")
MY_DRIVE = Path("/content/drive/MyDrive")
PROJECT = MY_DRIVE / "mediflow_Project"
BUNDLE = PROJECT / "MediFlow_Adaptation_Common_v3.zip"
if not BUNDLE.is_file():
    raise FileNotFoundError(f"공통 ZIP v3 위치를 확인하세요. {BUNDLE}")
EXPECTED = "4ae6e07eb4af122531439519ae2207aad57a1c8dfb8f158152a2093fa65a91fe"
OVERLAY = {'adaptation_stage1.py': '"""Compare two eligible starting models per domain with the same lighting treatment."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport math\nimport os\nimport shutil\nimport time\nimport uuid\nimport zipfile\nfrom collections import Counter\nfrom concurrent.futures import ThreadPoolExecutor\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nimport keras\nimport numpy as np\nimport tensorflow as tf\n\nfrom mediflow_datasets import camera_extended\nfrom mediflow_datasets import domain_adaptation_data as data\nfrom mediflow_datasets import domain_adaptation_experiment as exp\nfrom mediflow_datasets import domain_adaptation_resolution as resolution\nfrom mediflow_datasets.candidate_reproduction import ROOT\nfrom mediflow_datasets.domain_adaptation_io import write_progress\nfrom mediflow_datasets.domain_adaptation_training import object_hash\n\nDOMAINS = ("skin", "web_skin", "hair")\nARMS = ("basic", "selected")\nWORKERS = min(4, os.cpu_count() or 1)\n\n\ndef show(label, step, total, extra=""):\n    print(f"{label} | {step}/{total} ({100 * step / total:.1f}%) {extra}", flush=True)\n\n\ndef prepare_source(domain, archive_path, task, reference, out):\n    classes = task["classes"]\n    print(f"[{domain}] 데이터 ZIP 해시·클래스 확인", flush=True)\n    source, sha = data.source_index(archive_path, classes, task["source_sha256"])\n    retained = (\n        resolution.filtered_source(source, reference["quarantine"], classes, sha)\n        if reference.get("quarantine")\n        else {k: v for k, v in source.items() if k[0] == "original"}\n    )\n    # Skin\'s historical distinction is Original versus Augmented preparation. Use only\n    # linked descendants of retained originals. Adaptation itself uses identical originals.\n    train = retained["original", "train"]\n    originals = {r.path: r for r in train}\n    prepare_augmented = domain == "skin" and any(\n        spec.get("skin_augmented_preparation") for spec in task.get("models", {}).values()\n    )\n    descendants = [\n        r\n        for r in source["augmented", "train"]\n        if prepare_augmented\n        and r.origin in originals\n        and r.label == originals[r.origin].label\n        and r.path.rsplit("/", 1)[-1] != r.origin.rsplit("/", 1)[-1]\n    ]\n    # For the saved Hair model, include its previous augmented source in the leakage scan.\n    scanned = (\n        [r for rows in source.values() for r in rows]\n        if domain == "hair"\n        else [r for rows in retained.values() for r in rows] + descendants\n    )\n    label = f"[{domain}] 중복·분할 확인 (사진)"\n    entries = []\n    with zipfile.ZipFile(archive_path) as archive, ThreadPoolExecutor(WORKERS) as pool:\n\n        def signature(row):\n            return row, *data.image_signatures(archive.read(row.path))\n\n        for start in range(0, len(scanned), 128):\n            entries.extend(pool.map(signature, scanned[start : start + 128]))\n            done = len(entries)\n            if start == 0 or done % 1024 == 0 or done == len(scanned):\n                show(label, done, len(scanned))\n    report = data.audit_groups(entries)\n    dropped = set()\n    if descendants:\n        involved = {p for pair in report["near_duplicate_review"] for p in pair}\n        involved.update(\n            p for leak in report["blocking_leaks_or_conflicts"] for p in leak.get("paths", ())\n        )\n        dropped = {r.path for r in descendants if r.path in involved}\n        entries = [entry for entry in entries if entry[0].path not in dropped]\n        report = data.audit_groups(entries)\n    report["excluded_augmented_descendants"] = sorted(dropped)\n    report["source_target_pixel_overlap"] = len(\n        {entry[2] for entry in entries} & set(reference["pixel_hashes"])\n    )\n    report["source_sha256"] = sha\n    report["retained_counts"] = {split: len(retained["original", split]) for split in data.SPLITS}\n    if domain == "hair":\n        consistency = {}\n        for split in data.SPLITS:\n            original, augmented = (\n                Counter(\n                    (row.label, pixel)\n                    for row, _, pixel, _ in entries\n                    if row.split == split and row.variant == variant\n                )\n                for variant in ("original", "augmented")\n            )\n            consistency[split] = (\n                not (original - augmented) if split == "train" else original == augmented\n            )\n        report["original_augmented_consistency"] = consistency\n    write_progress(out / "data_audit.json", report)\n    if (\n        report["blocking_leaks_or_conflicts"]\n        or report["near_duplicate_review"]\n        or report["source_target_pixel_overlap"]\n        or not all(report.get("original_augmented_consistency", {}).values())\n    ):\n        raise ValueError(f"{domain} 분할 점검 실패. data_audit.json을 확인하세요.")\n    pixel_by_path = {row.path: pixel for row, _, pixel, _ in entries}\n    paired = {}\n    for row in descendants:\n        if row.path not in dropped and pixel_by_path[row.path] != pixel_by_path[row.origin]:\n            paired.setdefault(row.origin, row)\n    if prepare_augmented and {r.label for r in paired.values()} != set(range(len(classes))):\n        raise ValueError("피부 선정 방식 재현에 필요한 클래스별 파생본이 확인되지 않았습니다.")\n    write_progress(\n        out / "source_manifest.json",\n        {\n            "original": {\n                split: [asdict(r) for r in retained["original", split]] for split in data.SPLITS\n            },\n            "skin_preparation_pairs": {parent: asdict(row) for parent, row in paired.items()},\n        },\n    )\n    return train, retained["original", "val"], paired\n\n\ndef image_cache(folder, archive_path, train, val, pairs, task, size, profile, *, expanded=None):\n    """Cache exact float32 model inputs on local SSD once for each input resolution."""\n    rows = [*train, *val, *pairs.values()]\n    conditions = ("lighting", "expanded") if expanded is not None else ("lighting",)\n    size_bytes = (len(rows) + len(conditions) * len(val)) * size * size * 3 * 4\n    if shutil.disk_usage(folder).free < size_bytes + 1024**3:\n        raise RuntimeError(\n            f"{task[\'domain\']} 입력 캐시 공간 부족 (필요 {size_bytes / 1e9:.1f} GB)."\n        )\n    shape = (size, size, 3)\n    cache = np.lib.format.open_memmap(\n        folder / f"inputs_{size}.npy", mode="w+", dtype=np.float32, shape=(len(rows), *shape)\n    )\n    shifted = {\n        name: np.lib.format.open_memmap(\n            folder / f"val_{name}_{size}.npy", mode="w+", dtype=np.float32, shape=(len(val), *shape)\n        )\n        for name in conditions\n    }\n    with zipfile.ZipFile(archive_path) as archive, ThreadPoolExecutor(WORKERS) as pool:\n\n        def prepare(item):\n            i, row = item\n            with tf.device("/CPU:0"):\n                image = exp.image_tensor(archive.read(row.path), size, task["input_policy"])\n            cache[i] = image\n            if row.split == "val":\n                seed = exp.item_seed(row, 1707)\n                shifted["lighting"][i - len(train)] = exp.transform_image(\n                    image, profile, "target_guided", seed\n                )\n                if expanded is not None:\n                    shifted["expanded"][i - len(train)] = camera_extended.transform(\n                        image, expanded, "expanded", seed\n                    )\n\n        for start in range(0, len(rows), 128):\n            list(pool.map(prepare, enumerate(rows[start : start + 128], start)))\n            end = min(start + 128, len(rows))\n            if start == 0 or end % 512 == 0 or end == len(rows):\n                show(f"[{task[\'domain\']} · {size}px] 입력 캐시 준비 (사진)", end, len(rows))\n    cache.flush()\n    for images in shifted.values():\n        images.flush()\n    lookup = {row.path: i for i, row in enumerate(rows)}\n    paired_indexes = [\n        lookup[pairs[r.path].path] if r.path in pairs else i for i, r in enumerate(train)\n    ]\n    return cache, shifted if expanded is not None else shifted["lighting"], paired_indexes\n\n\ndef pipeline(images, indexes, labels, batch, size, transform=None):\n    def read(index):\n        image = images[int(index)]\n        return image if transform is None else transform(int(index), image)\n\n    def get(index, label):\n        image = tf.numpy_function(read, [index], tf.float32)\n        return tf.ensure_shape(image, (size, size, 3)), label\n\n    ds = tf.data.Dataset.from_tensor_slices(\n        (np.asarray(indexes, np.int32), np.asarray(labels, np.int32))\n    )\n    options = tf.data.Options()\n    options.threading.private_threadpool_size = WORKERS\n    options.threading.max_intra_op_parallelism = 1\n    return ds.with_options(options).map(get, num_parallel_calls=WORKERS).batch(batch).prefetch(2)\n\n\ndef model_for(spec, count, weights):\n    if spec["architecture"] == "pmg":\n        from mediflow_datasets.web_skin_paper_suite import build_pmg_model\n\n        return build_pmg_model(count, spec["size"], weights=weights)\n    size = spec["size"]\n    backbone = keras.applications.EfficientNetB0(\n        include_top=False, weights=weights, input_shape=(size, size, 3)\n    )\n    backbone.trainable = False\n    inputs = keras.Input((size, size, 3))\n    features = keras.layers.GlobalAveragePooling2D()(backbone(inputs, training=False))\n    outputs = keras.layers.Dense(count, activation="softmax")(keras.layers.Dropout(0.3)(features))\n    return keras.Model(inputs, outputs)\n\n\ndef predictor(model, spec):\n    pmg = spec["architecture"] == "pmg"\n    size = spec["size"]\n\n    @tf.function(input_signature=[tf.TensorSpec([None, size, size, 3], tf.float32)])\n    def predict(images):\n        outputs = model(images, training=False)\n        scores = tf.nn.softmax(tf.add_n(outputs), axis=-1) if pmg else outputs\n        return tf.debugging.check_numerics(scores, "validation scores")\n\n    return predict\n\n\ndef steps(model, spec, optimizer, count):\n    pmg = spec["architecture"] == "pmg"\n    loss_fn = keras.losses.CategoricalCrossentropy(\n        from_logits=pmg, label_smoothing=spec["label_smoothing"]\n    )\n    variables = model.trainable_variables\n    optimizer.build(variables)\n    if pmg:\n        from mediflow_datasets.web_skin_paper_suite import _jigsaw\n\n    def update(images, labels, branch=None, factor=1.0):\n        with tf.GradientTape() as tape:\n            outputs = model(images, training=True)\n            logits = outputs[branch] if pmg else outputs\n            loss = tf.debugging.check_numerics(factor * loss_fn(labels, logits), "training loss")\n        grads = tape.gradient(loss, variables)\n        pairs = [\n            (grad, var) for grad, var in zip(grads, variables, strict=True) if grad is not None\n        ]\n        clipped, norm = tf.clip_by_global_norm([g for g, _ in pairs], 1.0)\n        with tf.control_dependencies([tf.debugging.check_numerics(norm, "gradient norm")]):\n            optimizer.apply_gradients(zip(clipped, [v for _, v in pairs], strict=True))\n        return loss\n\n    signature = [\n        tf.TensorSpec([None, spec["size"], spec["size"], 3], tf.float32),\n        tf.TensorSpec([None], tf.int32),\n    ]\n\n    @tf.function(input_signature=signature)\n    def train_step(images, indices):\n        labels = tf.one_hot(indices, count)\n        if pmg:\n            branch_losses = [\n                update(_jigsaw(images, grid), labels, i) for i, grid in enumerate((8, 4, 2))\n            ]\n            branch_losses.append(update(images, labels, 3, 2.0))\n            return tf.add_n(branch_losses) / 4\n        return update(images, labels)\n\n    return train_step, predictor(model, spec)\n\n\ndef epoch_records(folder, epochs):\n    records = sorted(folder.glob("epoch_*.json"))\n    history = [data.read_json(p) for p in records]\n    if len(history) > epochs or [r["epoch"] for r in history] != list(range(1, len(history) + 1)):\n        raise ValueError(f"Incomplete or inconsistent epoch records: {folder}")\n    return history\n\n\ndef save_epoch(checkpoint, folder, local, entry):\n    attempt = "snapshot_" + uuid.uuid4().hex\n    destination = folder / attempt\n    destination.mkdir()\n    local.mkdir(exist_ok=True)\n    local_prefix = checkpoint.write(str(local / "state"))\n    record = {**entry, "snapshot": attempt, "files": {}}\n    for path in sorted(Path(local_prefix).parent.glob("state.*")):\n        shutil.copyfile(path, destination / path.name)\n        record["files"][path.name] = data.sha256(path)\n    if set(record["files"]) != {"state.index", "state.data-00000-of-00001"}:\n        raise RuntimeError("Unexpected checkpoint shard layout")\n    write_progress(folder / f"epoch_{entry[\'epoch\']:03}.json", record)\n    return record\n\n\ndef restore_epoch(checkpoint, folder, record):\n    for name, sha in record["files"].items():\n        if data.sha256(data.local_path(folder, f"{record[\'snapshot\']}/{name}")) != sha:\n            raise ValueError("Epoch checkpoint failed integrity verification")\n    checkpoint.read(str(folder / record["snapshot"] / "state")).assert_consumed()\n\n\ndef publish_model(source, directory, name):\n    temporary = directory / (uuid.uuid4().hex + ".keras.tmp")\n    shutil.copyfile(source, temporary)\n    destination = directory / f"{name}.keras"\n    temporary.replace(destination)\n    return {"file": destination.name, "sha256": data.sha256(source)}\n\n\ndef export_model(model, directory, name, local):\n    local.mkdir(exist_ok=True)\n    scratch = local / f"{name}.keras"\n    model.save(scratch)\n    return publish_model(scratch, directory, name)\n\n\ndef validation_pipelines(cache, shifted, n, val, batch, size):\n    labels = [row.label for row in val]\n    return {\n        "clean": pipeline(cache, range(n, n + len(val)), labels, batch, size),\n        **{\n            name: pipeline(images, range(len(val)), labels, batch, size)\n            for name, images in shifted.items()\n        },\n    }\n\n\ndef validate(predict, datasets, rows, class_count, label):\n    metrics = {}\n    for name, ds in datasets.items():\n        print(f"{label} Validation {name} · {len(rows):,}장", flush=True)\n        scores = np.concatenate([predict(images).numpy() for images, _ in ds])\n        metrics[name] = exp.classification(rows, scores, class_count)\n    return metrics\n\n\ndef phase(model, context, folder, local, spec, phase_name, epochs, rate):\n    config, task, train, val, profile, cache, shifted, pairs = context\n    folder.mkdir(exist_ok=True)\n    history = epoch_records(folder, epochs)\n    optimizer = keras.optimizers.Adam(rate)\n    step, predict = steps(model, spec, optimizer, len(task["classes"]))\n    checkpoint = tf.train.Checkpoint(model=model, optimizer=optimizer)\n    tag = f"[{task[\'domain\']} · {spec[\'role\']} · {phase_name}]"\n    if history:\n        restore_epoch(checkpoint, folder, history[-1])\n        print(f"{tag} Drive에서 {len(history)}/{epochs} epoch 복원", flush=True)\n    n, batch, size = len(train), task["batch_size"], spec["size"]\n    expanded = phase_name == "expanded"\n    condition = "expanded" if expanded else "lighting"\n    datasets = validation_pipelines(\n        cache, shifted if expanded else {"lighting": shifted}, n, val, batch, size\n    )\n    for epoch in range(len(history), epochs):\n        started = time.monotonic()\n        keras.utils.set_random_seed(config["seed"] + epoch)\n        order = np.random.default_rng(config["seed"] + epoch).permutation(n)\n        indexes = order\n        if phase_name.startswith("prepare") and spec.get("skin_augmented_preparation"):\n            indexes = [pairs[i] if (i + epoch) % 2 == 0 else i for i in order]\n        transform = None\n        if phase_name in ("lighting", "expanded"):\n            seeds = [exp.item_seed(row, config["seed"], epoch) for row in train]\n            apply = [exp.augmentation_applies(s, config["augmentation_probability"]) for s in seeds]\n\n            def transform(i, image, seeds=seeds, apply=apply):\n                if not apply[i]:\n                    return image\n                if expanded:\n                    return camera_extended.transform(image, profile, "expanded", seeds[i])\n                return exp.transform_image(image, profile, "target_guided", seeds[i])\n\n        label = f"{tag} Epoch {epoch + 1}/{epochs}"\n        batches = math.ceil(n / batch)\n        print(f"{label} 시작 · 학습 {n:,}장 · {batches} 배치", flush=True)\n        train_data = pipeline(\n            cache, indexes, [train[i].label for i in order], batch, size, transform\n        )\n        losses = []\n        for i, (images, labels) in enumerate(train_data, 1):\n            losses.append(float(step(images, labels)))\n            if i == 1 or i == batches or i % max(5, batches // 15) == 0:\n                elapsed = time.monotonic() - started\n                remaining = elapsed * (batches / i - 1)\n                show(\n                    label + " 학습",\n                    i,\n                    batches,\n                    f"loss={np.mean(losses):.5f} · 남은 학습 약 {remaining:.0f}초",\n                )\n        train_seconds = time.monotonic() - started\n        metrics = validate(predict, datasets, val, len(task["classes"]), label)\n        validation_seconds = time.monotonic() - started - train_seconds\n        entry = {\n            "epoch": epoch + 1,\n            "train_loss_mean_per_update": float(np.mean(losses)),\n            "training_batches": batches,\n            "optimizer_updates": batches * (4 if spec["architecture"] == "pmg" else 1),\n            "train_seconds": train_seconds,\n            "validation_seconds": validation_seconds,\n            **metrics,\n        }\n        print(f"{label} Drive 체크포인트 저장", flush=True)\n        history.append(save_epoch(checkpoint, folder, local, entry))\n        print(\n            f"{label} 완료 · loss={entry[\'train_loss_mean_per_update\']:.5f} · "\n            f"원본 Val F1={metrics[\'clean\'][\'macro_f1\']:.5f} · "\n            f"조명 Val F1={metrics[\'lighting\'][\'macro_f1\']:.5f}"\n            + (f" · 확장 Val F1={metrics[\'expanded\'][\'macro_f1\']:.5f}" if expanded else ""),\n            flush=True,\n        )\n    best = max(history, key=lambda r: score(r, phase_name in ("lighting", "expanded"), condition))\n    restore_epoch(checkpoint, folder, best)\n    write_progress(folder / "history.json", history)\n    return best\n\n\ndef score(record, lighting=True, condition="lighting"):\n    clean, shifted = record["clean"]["macro_f1"], record[condition]["macro_f1"]\n    return (min(clean, shifted), clean) if lighting else (clean,)\n\n\ndef trial(context, output, local, spec, job):\n    config, task = context[:2]\n    output.mkdir(exist_ok=True)\n    completed = output / "complete.json"\n    if completed.exists():\n        saved = data.read_json(completed)\n        verify_models(output, saved)\n        print(f"[{job}/6 · {task[\'domain\']} · {spec[\'role\']}] 완료 결과 재사용", flush=True)\n        return saved\n    print(f"\\n[{job}/6 · {task[\'domain\']} · {spec[\'role\']}] 실험 시작", flush=True)\n    keras.backend.clear_session()\n    keras.utils.set_random_seed(config["seed"])\n    count = len(task["classes"])\n    initial_path = output / "initial.json"\n    if initial_path.exists():\n        initial = data.read_json(initial_path)\n        path = data.local_path(output, initial["file"])\n        if data.sha256(path) != initial["sha256"]:\n            raise ValueError("Starting checkpoint changed")\n        model = keras.models.load_model(path, compile=False)\n    else:\n        if spec.get("model"):\n            path = data.local_path(ROOT, spec["model"])\n            if data.sha256(path) != spec["model_sha256"]:\n                raise ValueError("Selected checkpoint changed")\n            model = keras.models.load_model(path, compile=False)\n            initial = publish_model(path, output, "initial")\n            print(f"[{task[\'domain\']} · {spec[\'role\']}] 저장 가중치로 조명 적응 시작", flush=True)\n        else:\n            weights = None if (output / "prepare_head/epoch_001.json").exists() else "imagenet"\n            print(f"[{task[\'domain\']} · {spec[\'role\']}] 현재 Train으로 출발 모델 준비", flush=True)\n            model = model_for(spec, count, weights)\n            phase(\n                model,\n                context,\n                output / "prepare_head",\n                local,\n                spec,\n                "prepare_head",\n                config["prepare_head_epochs"],\n                config["prepare_head_lr"],\n            )\n            exp.configure_partial(model)\n            phase(\n                model,\n                context,\n                output / "prepare_partial",\n                local,\n                spec,\n                "prepare_partial",\n                config["prepare_partial_epochs"],\n                config["prepare_partial_lr"],\n            )\n            initial = export_model(model, output, "initial", local)\n        write_progress(initial_path, initial)\n    if tuple(model.input_shape[1:]) != (spec["size"], spec["size"], 3):\n        raise ValueError("Initial input shape differs from configuration")\n    if len(model.outputs) != (4 if spec["architecture"] == "pmg" else 1) or any(\n        layer.shape[-1] != count for layer in model.outputs\n    ):\n        raise ValueError("Initial output contract differs from class mapping")\n    exp.configure_partial(model)\n    best = phase(\n        model,\n        context,\n        output / "lighting",\n        local,\n        spec,\n        "lighting",\n        config["epochs"],\n        config["learning_rate"],\n    )\n    saved = {\n        "initial": initial,\n        "best_model": export_model(model, output, "best", local),\n        "model_config": spec,\n        "validation": best,\n        "source_test_evaluated": False,\n        "validation_scope": config.get("validation_scope", "current_source_validation"),\n    }\n    write_progress(completed, saved)\n    return saved\n\n\ndef verify_models(folder, result):\n    for key in ("initial", "best_model"):\n        item = result[key]\n        if data.sha256(data.local_path(folder, item["file"])) != item["sha256"]:\n            raise ValueError(f"Completed checkpoint changed: {folder}/{item[\'file\']}")\n\n\ndef run_domain(domain, source_zip, output, local):\n    config = data.read_json(ROOT / "configs/adaptation_stage1.json")\n    task = config["tasks"][domain]\n    if config.get("initialization_policy") == "saved_only" and any(\n        not spec.get("model") or not spec.get("model_sha256") for spec in task["models"].values()\n    ):\n        raise ValueError("A starting checkpoint is missing from the saved-only bundle")\n    if config.get("validation_scope"):\n        print(\n            f"[{domain}] 과거 학습 이력의 겹침 여부 미확인 · Validation은 탐색 비교용", flush=True\n        )\n    reference = data.read_json(ROOT / f"references/{domain}.json")\n    profile = reference["extended_profile"]["lighting"]\n    output.mkdir(parents=True, exist_ok=True)\n    identity = {\n        "config_sha256": object_hash(config),\n        "reference_sha256": object_hash(reference),\n        "code_sha256": data.sha256(Path(__file__)),\n    }\n    marker = output / "identity.json"\n    if marker.exists() and data.read_json(marker) != identity:\n        raise ValueError("Different inputs cannot resume this output")\n    write_progress(marker, identity)\n    write_progress(\n        output / "run_configuration.json",\n        {\n            "identity": identity,\n            "configuration": config,\n            "profile": profile,\n            "tensorflow": tf.__version__,\n            "keras": keras.__version__,\n            "numpy": np.__version__,\n            "gpu": [\n                tf.config.experimental.get_device_details(device)\n                for device in tf.config.list_physical_devices("GPU")\n            ],\n            "code_provenance": data.read_json(ROOT / "bundle_provenance.json"),\n        },\n    )\n    if all((output / role / "complete.json").is_file() for role in ARMS):\n        results = {role: data.read_json(output / role / "complete.json") for role in ARMS}\n        for role, result in results.items():\n            verify_models(output / role, result)\n        print(f"[{domain}] 두 학습 모두 완료됨", flush=True)\n        return results\n    train, val, pairs = prepare_source(domain, source_zip, task, reference, output)\n    caches = {}\n    results = {}\n    for i, role in enumerate(ARMS, 1):\n        existing = output / role / "complete.json"\n        if existing.is_file():\n            results[role] = data.read_json(existing)\n            verify_models(output / role, results[role])\n            print(\n                f"[{DOMAINS.index(domain) * 2 + i}/6 · {domain} · {role}] 완료 결과 재사용",\n                flush=True,\n            )\n            continue\n        spec = {**task["models"][role], "role": role}\n        size = spec["size"]\n        if size not in caches:\n            caches[size] = image_cache(local, source_zip, train, val, pairs, task, size, profile)\n        context = (config, task, train, val, profile, *caches[size])\n        results[role] = trial(\n            context, output / role, local / "saving", spec, DOMAINS.index(domain) * 2 + i\n        )\n    return results\n\n\ndef write_selection(root):\n    selected, table = {}, []\n    for domain in DOMAINS:\n        records = {role: data.read_json(root / domain / role / "complete.json") for role in ARMS}\n        for role, record in records.items():\n            verify_models(root / domain / role, record)\n            table.append(\n                {\n                    "domain": domain,\n                    "role": role,\n                    "best_epoch": record["validation"]["epoch"],\n                    "clean_macro_f1": record["validation"]["clean"]["macro_f1"],\n                    "lighting_macro_f1": record["validation"]["lighting"]["macro_f1"],\n                    "clean_accuracy": record["validation"]["clean"]["accuracy"],\n                    "lighting_accuracy": record["validation"]["lighting"]["accuracy"],\n                    "validation_scope": record.get("validation_scope", "current_source_validation"),\n                }\n            )\n        role = max(ARMS, key=lambda name: score(records[name]["validation"]))\n        record = records[role]\n        selected[domain] = {\n            "role": role,\n            "model_config": record["model_config"],\n            "initial_checkpoint": {\n                "path": f"{domain}/{role}/initial.keras",\n                "sha256": record["initial"]["sha256"],\n            },\n            "stage1_checkpoint": {\n                "path": f"{domain}/{role}/best.keras",\n                "sha256": record["best_model"]["sha256"],\n            },\n        }\n    selection = {\n        "schema": "mediflow_adaptation_stage1_selection_v1",\n        "selected": selected,\n        "criteria": "Maximize worst Val Macro-F1. Break ties by clean Val, then basic.",\n        "source_test_evaluated": False,\n        "target_accuracy": None,\n    }\n    if any(row["validation_scope"].startswith("exploratory_") for row in table):\n        selection["selection_scope"] = "provisional_followup_candidate"\n        selection["historical_train_evaluation_independence_verified"] = False\n    path = root / "selection.json"\n    if path.exists() and data.read_json(path) != selection:\n        raise ValueError("Existing Validation selection differs")\n    with (root / "validation_comparison.csv").open("w", encoding="utf-8-sig", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=table[0].keys())\n        writer.writeheader()\n        writer.writerows(table)\n    write_progress(path, selection)\n    for row in table:\n        print(row, flush=True)\n    print("1차 완료 · 분야별 선택:", {d: s["role"] for d, s in selected.items()}, flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--select", type=Path)\n    parser.add_argument("--domain", choices=DOMAINS)\n    parser.add_argument("--source-zip", type=Path)\n    parser.add_argument("--output", type=Path)\n    parser.add_argument("--local", type=Path)\n    args = parser.parse_args()\n    if args.select:\n        write_selection(args.select)\n    elif all((args.domain, args.source_zip, args.output, args.local)):\n        if not tf.config.list_physical_devices("GPU"):\n            raise RuntimeError("TensorFlow GPU를 사용할 수 없습니다.")\n        run_domain(args.domain, args.source_zip, args.output, args.local)\n    else:\n        parser.error("Provide --select or all domain run parameters")\n\n\nif __name__ == "__main__":\n    main()\n', 'adaptation_stage2.py': '"""Train three expanded camera treatments from the pinned Stage 1 starting models."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport sys\nimport tempfile\nimport uuid\nfrom pathlib import Path\n\nfrom mediflow_datasets import adaptation_stage1_colab as colab\nfrom mediflow_datasets import domain_adaptation_batch as batch\nfrom mediflow_datasets.domain_adaptation_io import write_progress\n\n\ndef read_json(path):\n    return json.loads(path.read_text(encoding="utf-8"))\n\n\ndef starting_plan(runtime, stage1, bundle_sha):\n    config = read_json(runtime / "configs/adaptation_stage1.json")\n    previous = read_json(stage1 / "bundle.json")\n    summary = read_json(stage1 / "summary.json")\n    selected = read_json(stage1 / "selection.json")\n    expected = {f"{d}/{role}" for d in colab.DOMAINS for role in ("basic", "selected")}\n    if (\n        previous["sha256"] != bundle_sha\n        or previous["stage"] != 1\n        or set(summary["completed_trials"]) != expected\n        or summary["failures"]\n        or not summary["selection_written"]\n        or selected["schema"] != "mediflow_adaptation_stage1_selection_v1"\n    ):\n        raise ValueError("공통 ZIP v3와 일치하는 1차 완료 결과가 필요합니다.")\n    plan = {}\n    for domain in colab.DOMAINS:\n        task = config["tasks"][domain]\n        choice = selected["selected"][domain]\n        role = choice["role"]\n        spec = {**task["models"][role], "role": role}\n        record = read_json(stage1 / domain / role / "complete.json")\n        initial = {"path": f"{domain}/{role}/initial.keras", "sha256": spec["model_sha256"]}\n        adapted = {"path": f"{domain}/{role}/best.keras", "sha256": record["best_model"]["sha256"]}\n        if (\n            choice["initial_checkpoint"] != initial\n            or choice["stage1_checkpoint"] != adapted\n            or choice["model_config"] != spec\n            or record["model_config"] != spec\n            or record["initial"]["sha256"] != initial["sha256"]\n        ):\n            raise ValueError(f"{domain}의 1차 선택 기록과 출발 모델이 일치하지 않습니다.")\n        plan[domain] = {"spec": spec, "stage1_best": adapted}\n    return config, plan\n\n\ndef run_domain(runtime, stage1, bundle_sha, domain, archive, output, local):\n    import cv2\n    import keras\n    import tensorflow as tf\n\n    from mediflow_datasets import adaptation_stage1 as train\n\n    config, plan = starting_plan(runtime, stage1, bundle_sha)\n    task = config["tasks"][domain]\n    spec, previous = plan[domain]["spec"], plan[domain]["stage1_best"]\n    reference = read_json(runtime / "references" / f"{domain}.json")\n    profile = reference["extended_profile"]\n    if profile["schema"] != "measured_camera_extended_v1" or profile["domain"] != domain:\n        raise ValueError(f"Unexpected expanded profile: {domain}")\n    original_path = train.data.local_path(runtime, spec["model"])\n    previous_path = train.data.local_path(stage1, previous["path"])\n    if batch.digest(original_path) != spec["model_sha256"] or (\n        batch.digest(previous_path) != previous["sha256"]\n    ):\n        raise ValueError(f"{domain} 모델 파일이 1차 기록과 일치하지 않습니다.")\n    source_manifest = stage1 / domain / "source_manifest.json"\n    identity = {\n        "bundle_sha256": bundle_sha,\n        "selection_sha256": batch.digest(stage1 / "selection.json"),\n        "source_manifest_sha256": batch.digest(source_manifest),\n        "reference_sha256": train.object_hash(reference),\n        "configuration_sha256": train.object_hash(config),\n        "code_sha256": {"stage2": batch.digest(__file__), "training": batch.digest(train.__file__)},\n    }\n    output.mkdir(parents=True, exist_ok=True)\n    marker = output / "identity.json"\n    if marker.exists() and read_json(marker) != identity:\n        raise ValueError(f"{domain} 기존 2차 결과와 이번 입력이 일치하지 않습니다.")\n    write_progress(marker, identity)\n    complete = output / "complete.json"\n    if complete.is_file():\n        result = read_json(complete)\n        if batch.digest(output / "best.keras") != result["best_model"]["sha256"]:\n            raise ValueError(f"{domain} 저장된 2차 모델이 완료 기록과 일치하지 않습니다.")\n        print(f"[{domain}] 2차 완료 결과 재사용", flush=True)\n        return\n\n    write_progress(\n        output / "run_configuration.json",\n        {\n            "stage": 2,\n            "identity": identity,\n            "training_configuration": config,\n            "model": spec,\n            "stage1_best": previous,\n            "treatment": "expanded",\n            "profile": profile,\n            "validation_conditions": ["clean", "lighting", "expanded"],\n            "validation_seed": 1707,\n            "tensorflow": tf.__version__,\n            "keras": keras.__version__,\n            "numpy": train.np.__version__,\n            "opencv": cv2.__version__,\n            "gpu": [\n                tf.config.experimental.get_device_details(d)\n                for d in tf.config.list_physical_devices("GPU")\n            ],\n            "bundle_provenance": read_json(runtime / "bundle_provenance.json"),\n        },\n    )\n    cv2.setNumThreads(1)\n    rows, val, pairs = train.prepare_source(domain, archive, task, reference, output)\n    if read_json(output / "source_manifest.json") != read_json(source_manifest):\n        raise ValueError(f"{domain}의 Train/Validation/Test 목록이 1차와 다릅니다.")\n    size = spec["size"]\n    cache, shifted, pair_indexes = train.image_cache(\n        local, archive, rows, val, pairs, task, size, profile["lighting"], expanded=profile\n    )\n    keras.utils.set_random_seed(config["seed"])\n    model = keras.models.load_model(original_path, compile=False)\n    count = len(task["classes"])\n    if tuple(model.input_shape[1:]) != (size, size, 3) or (\n        len(model.outputs) != (4 if spec["architecture"] == "pmg" else 1)\n        or any(tensor.shape[-1] != count for tensor in model.outputs)\n    ):\n        raise ValueError(f"{domain} 시작 모델의 입력 또는 클래스 설정이 다릅니다.")\n    train.exp.configure_partial(model)\n    print(f"[{domain} · {spec[\'role\']}] 1차 학습 전 저장본에서 확장 적응 시작", flush=True)\n    best = train.phase(\n        model,\n        (config, task, rows, val, profile, cache, shifted, pair_indexes),\n        output / "expanded",\n        local / "saving",\n        spec,\n        "expanded",\n        config["epochs"],\n        config["learning_rate"],\n    )\n    saved_model = train.export_model(model, output, "best", local / "saving")\n    del model\n    datasets = train.validation_pipelines(cache, shifted, len(rows), val, task["batch_size"], size)\n    benchmarks = {}\n    for name, path in (("initial", original_path), ("stage1_lighting", previous_path)):\n        keras.backend.clear_session()\n        model = keras.models.load_model(path, compile=False)\n        benchmarks[name] = train.validate(\n            train.predictor(model, spec), datasets, val, count, f"[{domain} · {name}]"\n        )\n        del model\n    benchmarks["stage2_expanded"] = {name: best[name] for name in datasets}\n    write_progress(\n        complete,\n        {\n            "best_model": saved_model,\n            "best_epoch": best["epoch"],\n            "model_config": spec,\n            "stage1_best": previous,\n            "validation": benchmarks,\n            "validation_scope": config["validation_scope"],\n            "source_test_evaluated": False,\n        },\n    )\n    print(f"[{domain}] 2차 완료 · 최적 {best[\'epoch\']}회차 · Drive 저장 완료", flush=True)\n\n\ndef write_comparison(destination):\n    rows = []\n    for domain in colab.DOMAINS:\n        complete = read_json(destination / domain / "complete.json")\n        for model, metrics in complete["validation"].items():\n            rows.append(\n                {\n                    "domain": domain,\n                    "start_role": complete["model_config"]["role"],\n                    "model": model,\n                    "validation_scope": complete["validation_scope"],\n                    **{f"{name}_macro_f1": result["macro_f1"] for name, result in metrics.items()},\n                    **{f"{name}_accuracy": result["accuracy"] for name, result in metrics.items()},\n                }\n            )\n    path = destination / "validation_comparison.csv"\n    temporary = path.with_name(f".{uuid.uuid4().hex}.csv.tmp")\n    with temporary.open("x", encoding="utf-8-sig", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n    temporary.replace(path)\n    return path\n\n\ndef launch(runtime, work, project, bundle_sha, my_drive):\n    import fcntl\n\n    parent = my_drive / "MediFlow_adaptation_runs" / bundle_sha[:16]\n    stage1, destination = parent / "stage1", parent / "stage2"\n    config, _ = starting_plan(runtime, stage1, bundle_sha)\n    lock = Path("/content") / f"mediflow_stage2_{bundle_sha}.lock"\n    with lock.open("a") as handle:\n        fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        python, versions = batch.prepare_python(work, executor=colab.run_live)\n        if not versions["gpu"]:\n            raise RuntimeError("Colab 런타임 유형에서 GPU를 선택한 뒤 다시 실행하세요.")\n        env = batch.python_environment(python, runtime)\n        env.update(PYTHONIOENCODING="utf-8", PYTHONDONTWRITEBYTECODE="1")\n        destination.mkdir(exist_ok=True)\n        write_progress(\n            destination / "bundle.json",\n            {\n                "sha256": bundle_sha,\n                "stage": 2,\n                "environment": versions,\n                "stage2_launcher_sha256": batch.digest(__file__),\n            },\n        )\n        print("2차 결과 저장:", destination, flush=True)\n        completed, failures = [], {}\n        for index, domain in enumerate(colab.DOMAINS, 1):\n            output = destination / domain\n            log = destination / f"{domain}_{uuid.uuid4().hex[:10]}.log"\n            print(f"\\n[{index}/3 · {domain}] 2차 준비", flush=True)\n            try:\n                with tempfile.TemporaryDirectory(prefix=f"{domain}_", dir=work) as folder:\n                    local = Path(folder)\n                    source = local / "source.zip"\n                    if not (output / "complete.json").is_file():\n                        drive_source = batch.find_source(\n                            project / "datasets", config["tasks"][domain]\n                        )\n                        colab.copy_source(drive_source, source)\n                    args = (runtime, stage1, bundle_sha, domain, source, output, local)\n                    command = [str(python), "-m", "mediflow_datasets.adaptation_stage2", "--domain"]\n                    colab.run_live(command + list(map(str, args)), env, log)\n                completed.append(domain)\n            except Exception as error:\n                failures[domain] = {"error": str(error), "log": log.name if log.is_file() else None}\n                print(f"[{domain}] 미완료: {error}", flush=True)\n        if not failures:\n            print("비교표:", write_comparison(destination), flush=True)\n        write_progress(\n            destination / "summary.json",\n            {\n                "completed_domains": completed,\n                "failures": failures,\n                "comparison_written": not failures,\n                "source_test_evaluated": False,\n            },\n        )\n        if failures:\n            raise RuntimeError(f"미완료 분야가 있습니다. 로그와 완료 기록: {destination}")\n        print("2차 세 분야 완료", flush=True)\n        return destination\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    mode = parser.add_mutually_exclusive_group(required=True)\n    mode.add_argument("--launch", nargs=5)\n    mode.add_argument("--domain", nargs=7)\n    args = parser.parse_args()\n    if args.launch:\n        runtime, work, project, sha, my_drive = args.launch\n        launch(Path(runtime), Path(work), Path(project), sha, Path(my_drive))\n        return\n    runtime, stage1, sha, domain, source, output, local = args.domain\n    run_domain(Path(runtime), Path(stage1), sha, domain, Path(source), Path(output), Path(local))\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'}

print("Drive 공통 ZIP v3 확인 중", BUNDLE, flush=True)
with tempfile.TemporaryDirectory(prefix="mediflow_stage2_", dir="/content") as temporary:
    work = Path(temporary)
    local_bundle = work / BUNDLE.name
    shutil.copyfile(BUNDLE, local_bundle)
    runtime = unpack_bundle(local_bundle, work / "runtime", EXPECTED)
    local_bundle.unlink()
    for name, module in OVERLAY.items():
        (runtime / "src/mediflow_datasets" / name).write_text(module, encoding="utf-8")
    env = {**os.environ, "PYTHONPATH": str(runtime / "src"), "PYTHONIOENCODING": "utf-8",
           "PYTHONDONTWRITEBYTECODE": "1"}
    command = [sys.executable, "-u", "-m", "mediflow_datasets.adaptation_stage2", "--launch",
               str(runtime), str(work), str(PROJECT), EXPECTED, str(MY_DRIVE)]
    with subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace",
                          start_new_session=True) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            if process.wait():
                raise RuntimeError("2차가 완료되지 않았습니다. 위 로그를 확인하세요.")
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)


In [ ]:

from pathlib import Path

from google.colab import drive
from IPython.display import Image, display

my_drive = Path("/content/drive/MyDrive")
if not my_drive.is_dir():
    drive.mount("/content/drive")
results = my_drive / "MediFlow_adaptation_runs" / "4ae6e07eb4af1225" / "stage2"
if not all((results / domain / "complete.json").is_file()
           for domain in ("skin", "web_skin", "hair")):
    raise FileNotFoundError("세 분야의 2차 학습을 완료한 뒤 이 셀을 실행하세요.")
plot_module = {"__name__": "mediflow_adaptation_curves"}
exec(compile('"""Plot completed camera-adaptation histories without loading models or datasets."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nDOMAINS = {"skin": "Microscope skin", "web_skin": "Webcam face", "hair": "Microscope scalp"}\nMODELS = {"basic": ("Basic", "#1766A6"), "selected": ("Selected", "#AD421E")}\nCONDITIONS = {\n    "clean": ("Original", "-", "o"),\n    "lighting": ("Lighting", "--", "s"),\n    "expanded": ("Expanded", ":", "^"),\n}\n\n\ndef read_json(path):\n    return json.loads(path.read_text(encoding="utf-8"))\n\n\ndef domain_curves(results, domain, stage):\n    folder = results / domain\n    trials = [(folder / role, role) for role in MODELS] if stage == 1 else [(folder, None)]\n    treatment = "lighting" if stage == 1 else "expanded"\n    curves = []\n    for trial, role in trials:\n        complete = read_json(trial / "complete.json")\n        history = read_json(trial / treatment / "history.json")\n        epochs = [row["epoch"] for row in history]\n        best = complete["validation"]["epoch"] if stage == 1 else complete["best_epoch"]\n        if epochs != list(range(1, len(epochs) + 1)) or best not in epochs:\n            raise ValueError(f"Incomplete epoch history: {trial}")\n        curves.append((role or complete["model_config"]["role"], epochs, history, best))\n    return curves\n\n\ndef save_curves(results, stage, output):\n    import matplotlib\n\n    matplotlib.use("Agg")\n    from matplotlib import pyplot as plt\n    from matplotlib.ticker import MaxNLocator, ScalarFormatter\n\n    if stage not in (1, 2):\n        raise ValueError("Expected Stage 1 or Stage 2")\n    results, output = Path(results), Path(output)\n    data = {domain: domain_curves(results, domain, stage) for domain in DOMAINS}\n    conditions = tuple(CONDITIONS)[: 2 if stage == 1 else 3]\n    fig, axes = plt.subplots(3, 3, figsize=(14, 9), layout="constrained")\n    experiment = "Lighting adaptation" if stage == 1 else "Expanded camera adaptation"\n    fig.suptitle(f"MediFlow  |  Stage {stage}  ·  {experiment}", x=0.02, ha="left", fontsize=18)\n    try:\n        for row, (domain, label) in enumerate(DOMAINS.items()):\n            loss_axis, f1_axis, accuracy_axis = axes[row]\n            loss_axis.set_ylabel(label, fontsize=11, fontweight="bold")\n            for role, epochs, history, best in data[domain]:\n                model_name, color = MODELS[role]\n                loss_axis.plot(\n                    epochs,\n                    [record["train_loss_mean_per_update"] for record in history],\n                    color=color,\n                    marker="o",\n                    label=model_name,\n                )\n                for axis, metric in ((f1_axis, "macro_f1"), (accuracy_axis, "accuracy")):\n                    for name in conditions:\n                        condition, style, marker = CONDITIONS[name]\n                        points = [record[name][metric] for record in history]\n                        axis.plot(\n                            epochs,\n                            points,\n                            color=color,\n                            linestyle=style,\n                            marker=marker,\n                            markersize=4,\n                            label=f"{model_name} · {condition}" if stage == 1 else condition,\n                        )\n                        axis.scatter(\n                            best,\n                            points[epochs.index(best)],\n                            facecolor="none",\n                            edgecolor=color,\n                            s=88,\n                            linewidth=1.2,\n                            zorder=5,\n                        )\n            for axis, title in zip(\n                axes[row],\n                ("Training loss", "Validation Macro-F1", "Validation accuracy"),\n                strict=True,\n            ):\n                if row == 0:\n                    axis.set_title(title, fontsize=12, fontweight="bold")\n                axis.xaxis.set_major_locator(MaxNLocator(integer=True))\n                axis.yaxis.set_major_locator(MaxNLocator(nbins=4, min_n_ticks=3))\n                axis.yaxis.set_major_formatter(ScalarFormatter(useOffset=False))\n                axis.grid(axis="y", alpha=0.22)\n                axis.spines[["top", "right"]].set_visible(False)\n                axis.tick_params(labelsize=9)\n                if row == len(DOMAINS) - 1:\n                    axis.set_xlabel("Epoch")\n            if stage == 2 or row == 0:\n                loss_axis.legend(loc="best", fontsize=8, frameon=False)\n                f1_axis.legend(loc="best", fontsize=7.5, framealpha=0.9)\n        fig.supxlabel(\n            "Rings mark the saved epoch. Validation axes are zoomed per panel. "\n            "Training loss definitions can vary by model.",\n            fontsize=9,\n            color="#475569",\n        )\n        output.parent.mkdir(parents=True, exist_ok=True)\n        fig.savefig(output, dpi=180, facecolor="white")\n    finally:\n        plt.close(fig)\n    return output\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--stage", type=int, choices=(1, 2), required=True)\n    parser.add_argument("--results", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    args = parser.parse_args()\n    print(save_curves(args.results, args.stage, args.output))\n\n\nif __name__ == "__main__":\n    main()\n', "adaptation_curves.py", "exec"), plot_module)
image_path = plot_module["save_curves"](results, 2, results / "stage2_learning_curves.png")
print("학습곡선 저장", image_path)
display(Image(filename=str(image_path)))
